# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [ ]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [ ]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

Now perform the same as before:
- Feature Scaling
- Feature Selection


In [ ]:
spaceship.shape

In [ ]:
spaceship.dtypes

In [ ]:
spaceship.isna().sum()

In [ ]:
spaceship = spaceship.dropna()
spaceship = spaceship.reset_index(drop=True)

In [ ]:
spaceship['Cabin'] = spaceship['Cabin'].apply(lambda x: x[0])
spaceship.Cabin.value_counts()

In [ ]:
spaceship = spaceship.drop(["PassengerId", "Name"], axis=1)
spaceship

**Perform Train Test Split**

In [ ]:
features = spaceship.drop(columns = "Transported")
target = spaceship["Transported"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(spaceship, target, test_size = 0.20, random_state=0)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False)

ohe.fit(X_train[['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP']])

In [ ]:
X_train_ohe = ohe.transform(X_train[['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP']])
X_test_ohe = ohe.transform(X_test[['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP']])
X_train_ohe

In [ ]:
X_train_ohe_df = pd.DataFrame(X_train_ohe, columns=ohe.get_feature_names_out(), index=X_train.index)
X_train_ohe_df

In [ ]:
X_test_ohe_df = pd.DataFrame(X_test_ohe, columns=ohe.get_feature_names_out(), index=X_test.index)
X_test_ohe_df

In [ ]:
X_train_num = X_train.select_dtypes(include=['number'])
X_train = pd.concat([X_train_num, X_train_ohe_df], axis = 1)
X_train

In [ ]:
X_test_num = X_test.select_dtypes(include=['number'])
X_test = pd.concat([X_test_num, X_test_ohe_df], axis = 1)
X_test

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

normalizer = MinMaxScaler()
normalizer.fit(X_train)

In [ ]:
X_train_norm = normalizer.transform(X_train)
X_train_norm = pd.DataFrame(X_train_norm, columns=X_train.columns, index=X_train.index )
X_train_norm.head()

In [ ]:
X_test_norm = normalizer.transform(X_test)
X_test_norm = pd.DataFrame(X_test_norm, columns=X_test.columns, index=X_test.index )
X_test_norm.head()

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [ ]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
bagging_clf = BaggingClassifier(DecisionTreeClassifier(max_depth=20),
                                n_estimators=100, # number of models to use
                                max_samples=1000,
                                random_state=42)
bagging_clf.fit(X_train_norm, y_train)

In [ ]:
y_pred_test_bag = bagging_clf.predict(X_test_norm)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
print(f"Accuracy: {accuracy_score(y_test, y_pred_test_bag):.2f}")
print(f"Precision: {precision_score(y_test, y_pred_test_bag):.2f}")
print(f"Recall: {recall_score(y_test, y_pred_test_bag):.2f}")
print(f"F1 Score: {f1_score(y_test, y_pred_test_bag):.2f}")

- Random Forests

In [ ]:
from sklearn.ensemble import RandomForestClassifier
forest_clf = RandomForestClassifier(n_estimators=100,
                             max_depth=20)
forest_clf.fit(X_train_norm, y_train)

In [ ]:
y_pred_test_forest = forest_clf.predict(X_test_norm)

print(f"Accuracy: {accuracy_score(y_test, y_pred_test_forest):.2f}")
print(f"Precision: {precision_score(y_test, y_pred_test_forest):.2f}")
print(f"Recall: {recall_score(y_test, y_pred_test_forest):.2f}")
print(f"F1 Score: {f1_score(y_test, y_pred_test_forest):.2f}")

- Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
gb_clf = GradientBoostingClassifier(n_estimators=100, max_depth=20)
gb_clf.fit(X_train_norm, y_train)

In [ ]:
y_pred_test_gb = gb_clf.predict(X_test_norm)

print(f"Accuracy: {accuracy_score(y_test, y_pred_test_gb):.2f}")
print(f"Precision: {precision_score(y_test, y_pred_test_gb):.2f}")
print(f"Recall: {recall_score(y_test, y_pred_test_gb):.2f}")
print(f"F1 Score: {f1_score(y_test, y_pred_test_gb):.2f}")

- Adaptive Boosting

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
ab_clf = AdaBoostClassifier(n_estimators=50)
ab_clf.fit(X_train_norm, y_train)

In [ ]:
y_pred_test_ab = ab_clf.predict(X_test_norm)

print(f"Accuracy: {accuracy_score(y_test, y_pred_test_ab):.2f}")
print(f"Precision: {precision_score(y_test, y_pred_test_ab):.2f}")
print(f"Recall: {recall_score(y_test, y_pred_test_ab):.2f}")
print(f"F1 Score: {f1_score(y_test, y_pred_test_ab):.2f}")

Which model is the best and why?

In [ ]:
# All models show quite similar performances. Since we want to solve a binary classifier, I would say the AdaBoost is the best model because of the highest F1 scorre.